# 08 — Ranking Model Evaluation

Compare CatBoost, XGBoost, and SVM as **ranking models** for lead prioritisation.
Each model is retrained from scratch using the exact hyperparameters from its
respective notebook (05, 06, 07). Models are evaluated on NDCG and AUC-ROC —
metrics that measure how well predicted probabilities rank leads by true value.

## Section 0 — Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import (
    OrdinalEncoder, LabelEncoder, StandardScaler,
)
from sklearn.metrics import (
    accuracy_score, ndcg_score, roc_auc_score,
)

SEED = 42
np.random.seed(SEED)
BAND_ORDER = ['Low', 'Medium', 'High']
RELEVANCE_MAP = {'Low': 0, 'Medium': 1, 'High': 2}

print('Imports complete.')

## Section 1 — Load Data & Map Relevance Scores

In [ ]:
X_train_raw = pd.read_csv('data/processed/X_train.csv')
X_test_raw  = pd.read_csv('data/processed/X_test.csv')
y_train     = pd.read_csv('data/processed/y_train.csv').squeeze()
y_test      = pd.read_csv('data/processed/y_test.csv').squeeze()
X_predict_raw = pd.read_csv('data/processed/X_predict.csv')

# Relevance scores for NDCG: Low=0, Medium=1, High=2
y_test_relevance = y_test.map(RELEVANCE_MAP).values

print(f'X_train: {X_train_raw.shape}')
print(f'X_test:  {X_test_raw.shape}')
print(f'X_predict: {X_predict_raw.shape}')
print(f'\nClass distribution (test):')
print(y_test.value_counts())

## Section 2 — Retrain CatBoost

Replicates notebook 05 preprocessing (feature engineering + native categoricals)
with the Optuna-tuned hyperparameters.

In [ ]:
# --- Feature engineering (identical to notebook 05) ---
AGE_ORDER = {
    'Under 25': 0, '25-34': 1, '35-44': 2,
    '45-54': 3, '55-64': 4, '65+': 5,
}
TIMELINE_URGENCY = {
    'Flexible': 0, '1 month': 1, '1-2 weeks': 2, 'ASAP': 3,
}
HIGH_VALUE_PROPERTY = {'Detached', 'Semi-Detached', 'Heritage'}


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Apply all feature engineering. Uses no target information (no leakage)."""
    df = df.copy()
    df['age_ordinal'] = df['customer_age_bracket'].map(AGE_ORDER).fillna(2).astype(int)
    df['timeline_urgency'] = df['requested_timeline'].map(TIMELINE_URGENCY).fillna(0).astype(int)

    sqft = df['estimated_job_size_sqft']
    q1, q2, q3 = sqft.quantile([0.25, 0.50, 0.75])

    def _size_tier(v):
        if v <= q1: return 'Small'
        if v <= q2: return 'Medium'
        if v <= q3: return 'Large'
        return 'XLarge'

    df['job_size_tier'] = sqft.apply(_size_tier)
    df['sqft_per_km'] = df['estimated_job_size_sqft'] / (df['distance_to_queens_km'] + 0.1)
    df['is_large_and_close'] = (
        (df['estimated_job_size_sqft'] >= q2) &
        (df['distance_to_queens_km'] <= df['distance_to_queens_km'].median())
    ).astype(int)
    df['is_homeowner_detached'] = (
        (df['homeowner_status'] == 'Own') &
        (df['property_type'].isin(HIGH_VALUE_PROPERTY))
    ).astype(int)
    return df


X_train_cb = engineer_features(X_train_raw)
X_test_cb  = engineer_features(X_test_raw)

CAT_FEATURES = [
    'property_type', 'neighbourhood', 'requested_timeline',
    'referral_source', 'homeowner_status', 'preferred_contact',
    'lead_capture_weather', 'customer_age_bracket',
    'lead_weekday', 'job_size_tier',
]

# Optuna-tuned hyperparameters from notebook 05
cb_params = {
    'iterations': 2000,
    'learning_rate': 0.1257934499324771,
    'depth': 4,
    'l2_leaf_reg': 4.2308261222393595,
    'random_strength': 3.0849453613732436,
    'bagging_temperature': 0.7713405860560613,
    'border_count': 63,
    'one_hot_max_size': 5,
    'grow_policy': 'Depthwise',
    'auto_class_weights': 'Balanced',
    'loss_function': 'MultiClass',
    'eval_metric': 'TotalF1:average=Macro',
    'random_seed': SEED,
    'verbose': 0,
}

print('Training CatBoost...')
model_cb = CatBoostClassifier(**cb_params)
model_cb.fit(
    X_train_cb, y_train,
    cat_features=CAT_FEATURES,
    eval_set=(X_test_cb, y_test),
    early_stopping_rounds=50,
)

# Predictions and probabilities
y_pred_cb = model_cb.predict(X_test_cb).flatten()
y_proba_cb = model_cb.predict_proba(X_test_cb)
cb_classes = list(model_cb.classes_)

acc_cb = accuracy_score(y_test, y_pred_cb)
print(f'CatBoost test accuracy: {acc_cb:.4f} ({acc_cb * 100:.1f}%)')

## Section 3 — Retrain XGBoost

Replicates notebook 06 preprocessing (OrdinalEncoder + LabelEncoder)
with the same hyperparameters.

In [ ]:
# --- Preprocessing (identical to notebook 06) ---
xgb_cat_features = [
    'property_type', 'neighbourhood', 'requested_timeline',
    'referral_source', 'homeowner_status', 'preferred_contact',
    'lead_capture_weather', 'customer_age_bracket', 'lead_weekday',
]

encoder_xgb = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoder_xgb.fit(X_train_raw[xgb_cat_features])

X_train_xgb = X_train_raw.copy()
X_test_xgb  = X_test_raw.copy()

X_train_xgb[xgb_cat_features] = encoder_xgb.transform(X_train_raw[xgb_cat_features]).astype(int)
X_test_xgb[xgb_cat_features]  = encoder_xgb.transform(X_test_raw[xgb_cat_features]).astype(int)

for df in [X_train_xgb, X_test_xgb]:
    df['has_pets'] = df['has_pets'].astype(int)

le_xgb = LabelEncoder()
y_train_enc = le_xgb.fit_transform(y_train)
y_test_enc  = le_xgb.transform(y_test)

print(f'Label mapping: {dict(zip(le_xgb.classes_, le_xgb.transform(le_xgb.classes_)))}')

# --- Train (identical hyperparameters from notebook 06) ---
print('\nTraining XGBoost...')
model_xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=SEED,
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    verbosity=0,
)
model_xgb.fit(
    X_train_xgb, y_train_enc,
    eval_set=[(X_test_xgb, y_test_enc)],
    verbose=False,
)

y_pred_xgb_enc = model_xgb.predict(X_test_xgb)
y_pred_xgb = le_xgb.inverse_transform(y_pred_xgb_enc)
y_proba_xgb = model_xgb.predict_proba(X_test_xgb)
xgb_classes = list(le_xgb.classes_)

acc_xgb = accuracy_score(y_test, y_pred_xgb)
print(f'XGBoost test accuracy: {acc_xgb:.4f} ({acc_xgb * 100:.1f}%)')

## Section 4 — Retrain SVM

Replicates notebook 07 preprocessing (drop nominal categoricals, ordinal-map
age/timeline, drop seasonal features, StandardScaler) with the tuned
hyperparameters (`kernel='linear'`, `C=100`).

In [ ]:
# --- Preprocessing (identical to notebook 07, Variant A + tuning) ---
svm_drop_cols = [
    'property_type', 'referral_source', 'homeowner_status',
    'preferred_contact', 'lead_capture_weather', 'lead_weekday',
]


def svm_encode(df: pd.DataFrame) -> pd.DataFrame:
    """Apply SVM-specific feature engineering from notebook 07."""
    df = df.drop(
        columns=[c for c in svm_drop_cols if c in df.columns], errors='ignore',
    ).copy()
    df['has_pets'] = df['has_pets'].map(
        {True: 1, False: 0, 'True': 1, 'False': 0},
    ).astype(int)

    age_map = {'18-24': 0, '25-34': 1, '35-44': 2, '45-54': 3, '55-64': 4, '65+': 5}
    df['customer_age_bracket'] = df['customer_age_bracket'].map(age_map).astype(int)

    timeline_map = {'ASAP': 2, '1-2 weeks': 10, '1 month': 30, 'Flexible': 90}
    df['timeline_log'] = df['requested_timeline'].map(timeline_map).apply(np.log1p)
    df = df.drop(columns=['requested_timeline'])
    return df


X_train_svm_shared = svm_encode(X_train_raw)
X_test_svm_shared  = svm_encode(X_test_raw)

# Variant A: drop neighbourhood
X_train_svm = X_train_svm_shared.drop(columns=['neighbourhood']).copy()
X_test_svm  = X_test_svm_shared.drop(columns=['neighbourhood']).copy()

# Drop seasonal features (per tuning section of notebook 07)
drop_seasonal = ['lead_month_sin', 'lead_month_cos']
X_train_svm = X_train_svm.drop(columns=drop_seasonal)
X_test_svm  = X_test_svm.drop(columns=drop_seasonal)

scaler_svm = StandardScaler()
X_train_svm_scaled = pd.DataFrame(
    scaler_svm.fit_transform(X_train_svm),
    columns=X_train_svm.columns, index=X_train_svm.index,
)
X_test_svm_scaled = pd.DataFrame(
    scaler_svm.transform(X_test_svm),
    columns=X_test_svm.columns, index=X_test_svm.index,
)

print(f'SVM features ({X_train_svm_scaled.shape[1]}): {list(X_train_svm_scaled.columns)}')

# --- Train (tuned hyperparameters from notebook 07) ---
print('\nTraining SVM...')
model_svm = SVC(
    kernel='linear',
    C=100,
    decision_function_shape='ovo',
    probability=True,
    random_state=SEED,
)
model_svm.fit(X_train_svm_scaled, y_train)

y_pred_svm = model_svm.predict(X_test_svm_scaled)
y_proba_svm = model_svm.predict_proba(X_test_svm_scaled)
svm_classes = list(model_svm.classes_)

acc_svm = accuracy_score(y_test, y_pred_svm)
print(f'SVM test accuracy: {acc_svm:.4f} ({acc_svm * 100:.1f}%)')

## Section 5 — Compute Ranking Metrics

For each model we compute:
- **NDCG** — measures whether leads with truly higher profit bands get
  higher ranking scores. Uses `prob_High` as the ranking score.
- **AUC-ROC** — measures class-separation quality across all three
  bands using OvR decomposition.
- **Accuracy** — for reference only (not a ranking metric).

In [ ]:
def get_prob_high(proba: np.ndarray, classes: list) -> np.ndarray:
    """Extract the prob_High column from a probability matrix."""
    high_idx = classes.index('High')
    return proba[:, high_idx]


def compute_ranking_metrics(
    name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
    classes: list,
    relevance: np.ndarray,
) -> dict:
    """
    Compute NDCG, AUC-ROC (macro, OvR), and accuracy for a model.

    Parameters
    ----------
    name : str
        Model name for display.
    y_true : pd.Series
        True labels (Low/Medium/High).
    y_pred : np.ndarray
        Predicted labels.
    y_proba : np.ndarray
        Probability matrix (n_samples × n_classes).
    classes : list
        Class order matching columns of y_proba.
    relevance : np.ndarray
        Integer relevance scores (Low=0, Medium=1, High=2).

    Returns
    -------
    dict
        Dictionary with model name and metric values.
    """
    prob_high = get_prob_high(y_proba, classes)

    # NDCG: reshape to (1, n_samples) for ndcg_score
    ndcg = ndcg_score(
        y_true=relevance.reshape(1, -1),
        y_score=prob_high.reshape(1, -1),
    )

    # AUC-ROC: reorder proba columns to match sorted unique labels
    sorted_labels = sorted(np.unique(y_true))
    ordered_proba = np.column_stack(
        [y_proba[:, classes.index(cls)] for cls in sorted_labels]
    )
    auc_roc = roc_auc_score(
        y_true, ordered_proba,
        multi_class='ovr', average='macro',
        labels=sorted_labels,
    )

    acc = accuracy_score(y_true, y_pred)

    return {
        'Model': name,
        'NDCG': ndcg,
        'AUC-ROC': auc_roc,
        'Accuracy': acc,
    }


results = [
    compute_ranking_metrics(
        'CatBoost', y_test, y_pred_cb, y_proba_cb, cb_classes, y_test_relevance,
    ),
    compute_ranking_metrics(
        'XGBoost', y_test, y_pred_xgb, y_proba_xgb, xgb_classes, y_test_relevance,
    ),
    compute_ranking_metrics(
        'SVM', y_test, y_pred_svm, y_proba_svm, svm_classes, y_test_relevance,
    ),
]

results_df = pd.DataFrame(results).set_index('Model')
print('Metrics computed for all models.')

## Section 6 — Comparison Table

In [ ]:
print('=' * 60)
print('       RANKING MODEL COMPARISON')
print('=' * 60)

display_df = results_df.copy()
for col in ['NDCG', 'AUC-ROC', 'Accuracy']:
    display_df[col] = display_df[col].apply(lambda x: f'{x:.4f}')

print(display_df.to_string())
print('=' * 60)

best_model_name = results_df['NDCG'].idxmax()
print(f'\n>>> Best model by NDCG: {best_model_name}')

## Section 7 — Bar Chart

Grouped bar chart showing NDCG and AUC-ROC side by side for each model,
with accuracy shown in a separate subplot for contrast.

In [ ]:
PALETTE = {'CatBoost': '#4C72B0', 'XGBoost': '#55A868', 'SVM': '#C44E52'}
models = results_df.index.tolist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [2, 1]})

# --- Left panel: NDCG and AUC-ROC grouped bars ---
x = np.arange(len(models))
bar_width = 0.35

ndcg_vals = results_df['NDCG'].values
auc_vals  = results_df['AUC-ROC'].values

bars1 = ax1.bar(x - bar_width / 2, ndcg_vals, bar_width, label='NDCG',
                color=[PALETTE[m] for m in models], alpha=0.9, edgecolor='white')
bars2 = ax1.bar(x + bar_width / 2, auc_vals, bar_width, label='AUC-ROC',
                color=[PALETTE[m] for m in models], alpha=0.55, edgecolor='white',
                hatch='//')

ax1.set_xlabel('Model', fontsize=12)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('NDCG & AUC-ROC by Model', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(models, fontsize=11)
ax1.legend(fontsize=11)
ax1.set_ylim(0, 1.05)

# Add value labels
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# --- Right panel: Accuracy ---
acc_vals = results_df['Accuracy'].values
bars3 = ax2.bar(models, acc_vals, color=[PALETTE[m] for m in models],
                alpha=0.9, edgecolor='white')
ax2.set_xlabel('Model', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Accuracy (Reference)', fontsize=14, fontweight='bold')
ax2.set_ylim(0, 1.05)

for bar in bars3:
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
Path('figures').mkdir(exist_ok=True)
fig.savefig('figures/18_ranking_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('Saved figures/18_ranking_metrics_comparison.png')

## Section 8 — Generate Ranked Leads

Using the best-performing model by NDCG, compute a weighted priority score
on `X_predict.csv`:

$$\text{weighted\_priority\_score} = (\text{prob\_High} \times 3) + (\text{prob\_Medium} \times 2) + (\text{prob\_Low} \times 1)$$

Sort descending and assign a `priority_rank` starting at 1.

In [ ]:
# --- Preprocess X_predict for the best model ---
if best_model_name == 'CatBoost':
    X_pred = engineer_features(X_predict_raw)
    proba_pred = model_cb.predict_proba(X_pred)
    pred_classes = cb_classes
elif best_model_name == 'XGBoost':
    X_pred = X_predict_raw.copy()
    X_pred[xgb_cat_features] = encoder_xgb.transform(X_predict_raw[xgb_cat_features]).astype(int)
    X_pred['has_pets'] = X_pred['has_pets'].astype(int)
    proba_pred = model_xgb.predict_proba(X_pred)
    pred_classes = xgb_classes
else:  # SVM
    X_pred = svm_encode(X_predict_raw)
    X_pred = X_pred.drop(columns=['neighbourhood']).copy()
    X_pred = X_pred.drop(columns=drop_seasonal)
    X_pred_scaled = pd.DataFrame(
        scaler_svm.transform(X_pred),
        columns=X_pred.columns, index=X_pred.index,
    )
    proba_pred = model_svm.predict_proba(X_pred_scaled)
    pred_classes = svm_classes

# Build results dataframe
ranked = X_predict_raw.copy()

prob_high_idx = pred_classes.index('High')
prob_med_idx  = pred_classes.index('Medium')
prob_low_idx  = pred_classes.index('Low')

ranked['prob_High']   = proba_pred[:, prob_high_idx]
ranked['prob_Medium'] = proba_pred[:, prob_med_idx]
ranked['prob_Low']    = proba_pred[:, prob_low_idx]

# Weighted priority score
ranked['weighted_priority_score'] = (
    ranked['prob_High'] * 3 +
    ranked['prob_Medium'] * 2 +
    ranked['prob_Low'] * 1
)

# Sort and rank
ranked = ranked.sort_values('weighted_priority_score', ascending=False).reset_index(drop=True)
ranked['priority_rank'] = range(1, len(ranked) + 1)

# Save
Path('data/outputs').mkdir(parents=True, exist_ok=True)
ranked.to_csv('data/outputs/ranked_leads.csv', index=False)

print(f'Best model: {best_model_name}')
print(f'Saved data/outputs/ranked_leads.csv ({len(ranked)} rows)')
print(f'\nTop 10 leads by priority:')
print(
    ranked[['priority_rank', 'weighted_priority_score',
            'prob_High', 'prob_Medium', 'prob_Low']].head(10).to_string(index=False)
)

## Section 9 — Precision@K Analysis

Precision@K answers the question: *"If the sales team calls only the top K leads
recommended by the model, what proportion are genuinely High-profit jobs?"*

This is the most operationally relevant metric — it directly measures how many
of the first K phone calls are worth making.

In [ ]:
def precision_at_k(
    y_true_relevance: np.ndarray,
    y_scores: np.ndarray,
    k: int,
) -> float:
    """
    Compute Precision@K for High-value leads.

    Parameters
    ----------
    y_true_relevance : np.ndarray
        Integer relevance scores (Low=0, Medium=1, High=2).
    y_scores : np.ndarray
        Ranking scores (higher = more likely High).
    k : int
        Number of top leads to consider.

    Returns
    -------
    float
        Proportion of top-K leads where true relevance == 2 (High).
    """
    top_k_idx = np.argsort(y_scores)[::-1][:k]
    return (y_true_relevance[top_k_idx] == 2).mean()


# Extract prob_High for each model
prob_high_cb  = get_prob_high(y_proba_cb, cb_classes)
prob_high_xgb = get_prob_high(y_proba_xgb, xgb_classes)
prob_high_svm = get_prob_high(y_proba_svm, svm_classes)

K_VALUES = [10, 20, 30, 50]

pak_rows = []
for k in K_VALUES:
    pak_rows.append({
        'K': k,
        'CatBoost': precision_at_k(y_test_relevance, prob_high_cb, k),
        'XGBoost':  precision_at_k(y_test_relevance, prob_high_xgb, k),
        'SVM':      precision_at_k(y_test_relevance, prob_high_svm, k),
    })

pak_df = pd.DataFrame(pak_rows).set_index('K')

# Baseline: proportion of High leads in test set
baseline_precision = (y_test_relevance == 2).mean()

print('=' * 60)
print('       PRECISION@K — HIGH-VALUE LEADS IN TOP K')
print('=' * 60)

display_pak = pak_df.copy()
for col in ['CatBoost', 'XGBoost', 'SVM']:
    display_pak[col] = display_pak[col].apply(lambda x: f'{x:.2%}')

print(display_pak.to_string())
print(f'\nBaseline (random): {baseline_precision:.2%}')
print('=' * 60)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for model_name in ['CatBoost', 'XGBoost', 'SVM']:
    ax.plot(
        K_VALUES, pak_df[model_name].values,
        marker='o', linewidth=2, markersize=7,
        color=PALETTE[model_name], label=model_name,
    )

ax.axhline(
    y=baseline_precision, color='grey', linestyle='--',
    linewidth=1.5, label=f'Baseline ({baseline_precision:.1%})',
)

ax.set_xlabel('K (number of top leads called)', fontsize=12)
ax.set_ylabel('Precision@K', fontsize=12)
ax.set_title(
    'Precision@K — How Many Top Leads Are Truly High Value?',
    fontsize=14, fontweight='bold',
)
ax.set_xticks(K_VALUES)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig('figures/19_precision_at_k.png', dpi=150, bbox_inches='tight')
plt.show()

print('Saved figures/19_precision_at_k.png')

In [ ]:
# Find the best (model, K) combination by highest Precision@K
best_k = None
best_model = None
best_pak = 0.0

for k in K_VALUES:
    for model_name in ['CatBoost', 'XGBoost', 'SVM']:
        val = pak_df.loc[k, model_name]
        if val > best_pak:
            best_pak = val
            best_k = k
            best_model = model_name

print(
    f'When the sales team calls the top {best_k} leads recommended by '
    f'{best_model}, {best_pak:.0%} are genuinely High profit jobs, '
    f'compared to {baseline_precision:.0%} if leads were called at random.'
)

## Section 10 — Recall@K Analysis

Recall@K answers the complementary question: *"Of all the genuinely High-profit
leads in the test set, how many did the model surface in its top K
recommendations?"*

High Precision@K means the calls you make are worthwhile;
high Recall@K means you are not leaving money on the table.

In [ ]:
def recall_at_k(
    y_true_relevance: np.ndarray,
    y_scores: np.ndarray,
    k: int,
) -> float:
    """
    Compute Recall@K for High-value leads.

    Parameters
    ----------
    y_true_relevance : np.ndarray
        Integer relevance scores (Low=0, Medium=1, High=2).
    y_scores : np.ndarray
        Ranking scores (higher = more likely High).
    k : int
        Number of top leads to consider.

    Returns
    -------
    float
        Proportion of all High leads captured in the top K.
    """
    total_high = (y_true_relevance == 2).sum()
    if total_high == 0:
        return 0.0
    top_k_idx = np.argsort(y_scores)[::-1][:k]
    high_in_top_k = (y_true_relevance[top_k_idx] == 2).sum()
    return high_in_top_k / total_high


rak_rows = []
for k in K_VALUES:
    rak_rows.append({
        'K': k,
        'CatBoost': recall_at_k(y_test_relevance, prob_high_cb, k),
        'XGBoost':  recall_at_k(y_test_relevance, prob_high_xgb, k),
        'SVM':      recall_at_k(y_test_relevance, prob_high_svm, k),
    })

rak_df = pd.DataFrame(rak_rows).set_index('K')

total_high = (y_test_relevance == 2).sum()

# --- Combined Precision@K and Recall@K table ---
print('=' * 70)
print('       PRECISION@K vs RECALL@K — SIDE BY SIDE')
print('=' * 70)
print()

header = f"{'K':>3s}  {'':^30s}  │  {'':^30s}"
print(f"{'':>3s}  {'Precision@K':^30s}  │  {'Recall@K':^30s}")
print(f"{'K':>3s}  {'CatBoost':>9s}  {'XGBoost':>9s}  {'SVM':>9s}  │  "
      f"{'CatBoost':>9s}  {'XGBoost':>9s}  {'SVM':>9s}")
print('-' * 70)

for k in K_VALUES:
    p_cb  = pak_df.loc[k, 'CatBoost']
    p_xgb = pak_df.loc[k, 'XGBoost']
    p_svm = pak_df.loc[k, 'SVM']
    r_cb  = rak_df.loc[k, 'CatBoost']
    r_xgb = rak_df.loc[k, 'XGBoost']
    r_svm = rak_df.loc[k, 'SVM']
    print(f'{k:3d}  {p_cb:>9.2%}  {p_xgb:>9.2%}  {p_svm:>9.2%}  │  '
          f'{r_cb:>9.2%}  {r_xgb:>9.2%}  {r_svm:>9.2%}')

print('-' * 70)
print(f'Total High leads in test set: {total_high}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for model_name in ['CatBoost', 'XGBoost', 'SVM']:
    ax.plot(
        K_VALUES, rak_df[model_name].values,
        marker='o', linewidth=2, markersize=7,
        color=PALETTE[model_name], label=model_name,
    )

# Perfect recall line: if the model ranked all High leads first,
# recall at K = min(K, total_high) / total_high
perfect_recall = [min(k, total_high) / total_high for k in K_VALUES]
ax.plot(
    K_VALUES, perfect_recall,
    color='grey', linestyle='--', linewidth=1.5,
    label='Perfect ranking',
)

ax.set_xlabel('K (number of top leads called)', fontsize=12)
ax.set_ylabel('Recall@K', fontsize=12)
ax.set_title(
    'Recall@K — How Many High Value Leads Are We Missing?',
    fontsize=14, fontweight='bold',
)
ax.set_xticks(K_VALUES)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig('figures/20_recall_at_k.png', dpi=150, bbox_inches='tight')
plt.show()

print('Saved figures/20_recall_at_k.png')

In [ ]:
# Find the best (model, K) combination by highest Recall@K
best_recall_k = None
best_recall_model = None
best_recall_val = 0.0

for k in K_VALUES:
    for model_name in ['CatBoost', 'XGBoost', 'SVM']:
        val = rak_df.loc[k, model_name]
        if val > best_recall_val:
            best_recall_val = val
            best_recall_k = k
            best_recall_model = model_name

missed_pct = (1 - best_recall_val) * 100

print(
    f'By calling the top {best_recall_k} leads, the sales team captures '
    f'{best_recall_val:.0%} of all genuinely High profit jobs available, '
    f'missing {missed_pct:.0f}% of High value opportunities.'
)

## Key Takeaways

- **NDCG** rewards models whose probability estimates correctly rank High leads above Medium, and Medium above Low — directly aligned with the business goal of lead prioritisation.
- **AUC-ROC (macro, OvR)** measures each class's separability from the rest, averaged across all three bands.
- The best-performing model by NDCG was used to generate `ranked_leads.csv`, which sales staff can sort by `priority_rank` to call the highest-value leads first.